# Greedy Experiments

Write your experiments in this document. The experiments below explore greedy Gray code constructions for **integer partitions** of n.

# Integer Partition Experiments

This section explores greedy Gray code constructions for integer partitions of n. We try two types of operations:
- **Move-one-unit**: move 1 unit from one part to another (can create or destroy parts of size 1)
- **Split/merge**: split one part into two, or merge two parts into one

For each operation type we experiment with:
- Different **starting partitions**: `(n,)` (single large part) vs `(1,1,...,1)` (all ones)
- Different **priority rules**: lexicographically smallest vs lexicographically largest next partition

Key findings:
- Move-one-unit with lex-min priority starting from `(1,...,1)` works for small n but fails at n=6.
- **Split/merge with lex-min priority starting from `(1,...,1)` works for n=1 through n=7** but fails at n=8 (missing only the partition `(7,1)`).
- Split/merge with fewest-parts priority starting from `(1,...,1)` also gets very close (misses by 1 for n=8,9,10,11).

## Objects: Integer Partitions

In [ ]:
# The set of all integer partitions of n.
# Each partition is represented as a tuple in weakly decreasing order.
# For example, the partitions of 4 are: (4,), (3,1), (2,2), (2,1,1), (1,1,1,1).
def allPartitions(n):
  S = set()
  def helper(remaining, maxPart, current):
    if remaining == 0:
      S.add(tuple(current))
      return
    for part in range(min(remaining, maxPart), 0, -1):
      helper(remaining - part, part, current + [part])
  helper(n, n, [])
  return S

In [ ]:
# Test the set of all objects.
n = 5
S = allPartitions(n)
for p in sorted(S):
  print(p)
print("total: %d" % len(S))

## Operations: Move-One-Unit and Split/Merge

In [ ]:
# Move-one-unit: move 1 unit from one part to another.
# If a part of size 1 loses its unit, that part is removed.
# A unit can also move into a brand-new part of size 1.
def neighborsMove(partition):
  parts = list(partition)
  results = set()
  n = sum(parts)
  for i in range(len(parts)):
    for j in range(len(parts)):
      if i == j:
        continue
      new_parts = parts[:]
      new_parts[i] -= 1
      new_parts[j] += 1
      new_parts = [p for p in new_parts if p > 0]
      new_parts_sorted = tuple(sorted(new_parts, reverse=True))
      if new_parts_sorted and sum(new_parts_sorted) == n:
        results.add(new_parts_sorted)
    # Split off a new part of size 1 from part i
    if parts[i] > 1:
      new_parts = parts[:]
      new_parts[i] -= 1
      new_parts.append(1)
      results.add(tuple(sorted(new_parts, reverse=True)))
  return results - {partition}

# Split/merge: split one part into two positive parts, or merge two parts into one.
def neighborsSplitMerge(partition):
  parts = list(partition)
  results = set()
  # Splits
  for i in range(len(parts)):
    for a in range(1, parts[i]):
      b = parts[i] - a
      new_parts = parts[:i] + parts[i+1:] + [a, b]
      results.add(tuple(sorted(new_parts, reverse=True)))
  # Merges
  for i in range(len(parts)):
    for j in range(i+1, len(parts)):
      merged = parts[i] + parts[j]
      new_parts = [parts[k] for k in range(len(parts)) if k != i and k != j] + [merged]
      results.add(tuple(sorted(new_parts, reverse=True)))
  return results - {partition}

In [ ]:
# Testing the operations
print("Move-one-unit neighbors of (3, 2, 1):")
for p in sorted(neighborsMove((3, 2, 1))):
  print(" ", p)

print("\nSplit/merge neighbors of (3, 2, 1):")
for p in sorted(neighborsSplitMerge((3, 2, 1))):
  print(" ", p)

## Experiments

In [ ]:
# Generic greedy algorithm for integer partitions.
# Inputs: n (integer to partition), start (starting partition),
#         neighborsFunc (function returning neighbors),
#         priority_key (key function for choosing next partition).
def greedyPartition(n, start, neighborsFunc, priority_key):
  visited = set()
  current = start
  visited.add(current)
  yield current
  while True:
    candidates = [c for c in neighborsFunc(current) if c not in visited]
    if not candidates:
      break
    current = min(candidates, key=priority_key)
    visited.add(current)
    yield current

In [ ]:
# --- MOVE-ONE-UNIT EXPERIMENTS ---

# Experiment 0: Move-one-unit, start=(n,), priority=lex min
def greedyMoveLexMin_startN(n):
  yield from greedyPartition(n, (n,), neighborsMove, lambda p: p)

# Experiment 1: Move-one-unit, start=(n,), priority=lex max
def greedyMoveLexMax_startN(n):
  yield from greedyPartition(n, (n,), neighborsMove, lambda p: tuple(-x for x in p))

# Experiment 2: Move-one-unit, start=(1,...,1), priority=lex min
def greedyMoveLexMin_startOnes(n):
  yield from greedyPartition(n, (1,)*n, neighborsMove, lambda p: p)

# Experiment 3: Move-one-unit, start=(1,...,1), priority=lex max
def greedyMoveLexMax_startOnes(n):
  yield from greedyPartition(n, (1,)*n, neighborsMove, lambda p: tuple(-x for x in p))

In [ ]:
# --- SPLIT/MERGE EXPERIMENTS ---

# Experiment 4: Split/merge, start=(n,), priority=lex min
def greedySMLexMin_startN(n):
  yield from greedyPartition(n, (n,), neighborsSplitMerge, lambda p: p)

# Experiment 5: Split/merge, start=(n,), priority=lex max
def greedySMLexMax_startN(n):
  yield from greedyPartition(n, (n,), neighborsSplitMerge, lambda p: tuple(-x for x in p))

# Experiment 6: Split/merge, start=(1,...,1), priority=lex min
# This is the most promising experiment!
def greedySMLexMin_startOnes(n):
  yield from greedyPartition(n, (1,)*n, neighborsSplitMerge, lambda p: p)

# Experiment 7: Split/merge, start=(1,...,1), priority=lex max
def greedySMLexMax_startOnes(n):
  yield from greedyPartition(n, (1,)*n, neighborsSplitMerge, lambda p: tuple(-x for x in p))

# Experiment 8: Split/merge, start=(1,...,1), priority=fewest parts (then lex min)
def greedySM_fewestParts_startOnes(n):
  yield from greedyPartition(n, (1,)*n, neighborsSplitMerge, lambda p: (len(p), p))

# Experiment 9: Split/merge, start=(1,...,1), priority=most parts (then lex min)
def greedySM_mostParts_startOnes(n):
  yield from greedyPartition(n, (1,)*n, neighborsSplitMerge, lambda p: (-len(p), p))

## Results

In [ ]:
# Run all experiments for n between nMin and nMax.
experiments = [
  greedyMoveLexMin_startN,
  greedyMoveLexMax_startN,
  greedyMoveLexMin_startOnes,
  greedyMoveLexMax_startOnes,
  greedySMLexMin_startN,
  greedySMLexMax_startN,
  greedySMLexMin_startOnes,
  greedySMLexMax_startOnes,
  greedySM_fewestParts_startOnes,
  greedySM_mostParts_startOnes,
]
nMin = 1
nMax = 10

for experimentNum, experiment in enumerate(experiments):
  print("\n\n" + "Experiment %d: " % experimentNum + experiment.__name__)

  results = []
  for n in range(nMin, nMax+1):
    print("\nn = %d" % n)
    allObjects = allPartitions(n)
    goal = len(allObjects)
    objects = []
    for partition in experiment(n):
      print(partition)
      objects.append(partition)
    numObjects = len(objects)
    print("total: %d / %d" % (numObjects, goal))
    result = (set(objects) == set(allObjects))
    results.append(result)

  print("\nSummary of " + experiment.__name__ + ":")
  numExperiments = nMax - nMin + 1
  for i in range(numExperiments):
    n = i + nMin
    print("n = %d: %s" % (n, results[i]))